# Fine-tune complet Omnilingual 1B — baoulé (GPU loué)

Décision : ADR-0044 · issue #474 · suite du test mécanique `docs/benchmarks/0005`.

**But** : entraîner **complètement** `omniASR_CTC_1B` (le meilleur modèle de départ : 22,1 % d'erreur par caractère
au benchmark 0003) sur Common Voice `bci` + la dictée d'esse, puis mesurer le gain **dans les mêmes conditions**.

**Deux temps, pour ne jamais payer à l'aveugle :**
1. **MESURE** (cellule 8) — 100 étapes, ~1 h : vérifie que tout tourne et mesure la vitesse → coût exact.
2. **COMPLET** (cellule 9) — le nombre d'étapes est proposé pour rester **dans le budget** (`BUDGET_USD`).

**Mesure** : à chaque validation, deux jeux sont évalués :
- `dev` (267 clips) → sert à **choisir la meilleure version** ;
- `test` (290 clips, ceux du benchmark 0003, **jamais entraînés**) → le chiffre à comparer. La validation au départ
  donne le score du modèle **non entraîné**, dans exactement les mêmes conditions.

**Machine (RunPod)** : 1× **A100 80 GB**, filtre **CUDA ≥ 12.8**, template **PyTorch** (avec Jupyter),
**Volume disk 100 GB** (monté sur `/workspace`). Avant de lancer : téléverse `dictee_bci_433.zip` dans `/workspace`
(glisser-déposer dans le panneau de fichiers de Jupyter).

> ⚠️ La machine est **facturée à l'heure tant qu'elle existe**. À la fin : télécharge les résultats, puis
> **Terminate** le pod (voir la dernière cellule).

In [ ]:
# Cellule 1 — chemins, budget, lanceur commun (re-exécuter si le noyau redémarre)
import os, re, shutil, subprocess, sys, time
from pathlib import Path

WORK = Path("/workspace") if Path("/workspace").is_dir() else Path.cwd()
os.chdir(WORK)
VENV = WORK / ".venv-omni312"          # sur le volume : survit à un redémarrage du pod
VENV_PY = str(VENV / "bin" / "python")
# Caches sur le volume (modèles, dataset) : pas de re-téléchargement après un redémarrage
os.environ.setdefault("HF_HOME", str(WORK / ".cache" / "huggingface"))
os.environ.setdefault("FAIRSEQ2_CACHE_DIR", str(WORK / ".cache" / "fairseq2"))
# RunPod active HF_HUB_ENABLE_HF_TRANSFER=1 mais hf_transfer n'est pas installé dans notre Python 3.12 :
# le téléchargement Hugging Face échouerait (constaté le 2026-10-19). Téléchargement normal.
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

# ---- À REMPLIR : prix horaire affiché par RunPod pour ta machine, et budget total ----
PRIX_HEURE_USD = 1.8                   # A100 SXM 80 GB RunPod le 2026-10-19
BUDGET_USD = 15.0
if "SESSION_T0" not in globals():
    SESSION_T0 = time.time()           # début approximatif de la facturation vue d'ici

def depense_estimee():
    return (time.time() - SESSION_T0) / 3600 * PRIX_HEURE_USD

def clean_env(extra=None):
    """Environnement des sous-processus SANS jetons (fairseq2 journalise toutes les variables)."""
    env = {k: v for k, v in os.environ.items()
           if not any(s in k.upper() for s in ("TOKEN", "SECRET", "PASSWORD", "CREDENTIAL", "API_KEY"))}
    env["PYTHONUNBUFFERED"] = "1"
    env.update(extra or {})
    return env

def sh(args, cwd=None, env=None, log_path=None, check=True, input_text=None, show=None):
    """Lance une commande, affiche sa sortie en direct (copiée en entier dans log_path), retourne le code.

    `show` (regex) : n'affiche que les lignes qui correspondent. Un entraînement écrit des milliers
    de lignes : tout afficher a saturé la mémoire du navigateur (« Out of Memory », 2026-10-19)."""
    proc = subprocess.Popen([str(a) for a in args], cwd=cwd, env=env or clean_env(),
                            stdin=subprocess.PIPE if input_text else None, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    if input_text:
        proc.stdin.write(input_text)
        proc.stdin.close()
    log = open(log_path, "w", encoding="utf-8") if log_path else None
    for line in proc.stdout:
        if show is None or show.search(line):
            print(line, end="")
        if log:
            log.write(line)
    if log:
        log.close()
    code = proc.wait()
    if check and code != 0:
        raise RuntimeError(f"ECHEC (code {code}) : {' '.join(str(a) for a in args)[:200]}")
    return code

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout or "PAS DE GPU")
print("Dossier de travail :", WORK, "| disque libre :", round(shutil.disk_usage(WORK).free / 2**30), "Gio")
print(f"Budget : {BUDGET_USD} USD à {PRIX_HEURE_USD} USD/h ≈ {BUDGET_USD / PRIX_HEURE_USD:.1f} h de machine")

## Étape 1 — Installation (Python 3.12 dédié, ~5–10 min)

Mêmes versions que les essais Kaggle (`omnilingual-asr==0.1.0`, `fairseq2[arrow]==0.6`, `torch==2.8.0`) + `tensorboard`
(activé par la recette, jamais installé par elle) + `ffmpeg` (décodage du webm de la dictée).

In [ ]:
# Cellule 2 — ffmpeg + Python 3.12 (uv) + versions figées + recette au même tag
if shutil.which("ffmpeg") is None:
    sh(["apt-get", "update", "-qq"])
    sh(["apt-get", "install", "-y", "-qq", "ffmpeg"])
UV = [sys.executable, "-m", "uv"]
sh([sys.executable, "-m", "pip", "install", "--quiet", "uv", "pyyaml"])
if not Path(VENV_PY).exists():
    sh(UV + ["venv", "--quiet", "--python", "3.12", VENV])
sh(UV + ["pip", "install", "--quiet", "--no-cache", "--python", VENV_PY,
         "omnilingual-asr==0.1.0", "fairseq2[arrow]==0.6", "torch==2.8.0", "torchaudio==2.8.0",
         "datasets<4.0", "soundfile", "librosa<1.0", "pyyaml", "tensorboard",
         # pyarrow récent exige NumPy 2, alors que l'environnement résolu garde NumPy 1.26 (2026-10-19).
         # 20.0.0 = minimum demandé par omnilingual-asr, compatible NumPy 1.26.
         "pyarrow==20.0.0"])
if not Path("omnilingual-asr").is_dir():
    sh(["git", "clone", "--quiet", "--depth", "1", "--branch", "0.1.0",
        "https://github.com/facebookresearch/omnilingual-asr.git"])
# Correctif du code Meta (wer_calculator.py, 0.1.0) : une transcription VIDE (le 1B ne sort que des
# « blancs » sur quelques clips) donne une longueur 0 et fait planter la validation. On la remplace par
# 1 symbole de remplissage : l'erreur calculée est identique (n erreurs pour une référence de n unités).
_wer = Path("omnilingual-asr/workflows/recipes/wav2vec2/asr/wer_calculator.py")
_src = _wer.read_text(encoding="utf-8")
_old = "            hyp_seqs.append(hyp_seq)\n"
if "correctif Wourri" not in _src:
    if _src.count(_old) != 1:
        raise RuntimeError("wer_calculator.py a changé : correctif à revoir avant d'entraîner")
    _wer.write_text(_src.replace(_old,
        "            if hyp_seq.numel() == 0:  # correctif Wourri : transcription vide\n"
        "                hyp_seq = hyp_seq.new_full((1,), self._pad_idx)\n" + _old), encoding="utf-8")
sh([VENV_PY, "-c", "import sys, torch; print('Python', sys.version.split()[0], '| torch', torch.__version__, "
    "'| GPU', torch.cuda.get_device_name(0), '| bf16', torch.cuda.is_bf16_supported())"])
print(f"\n>>> INSTALLATION TERMINÉE -> cellule 3. Dépense estimée : {depense_estimee():.2f} USD")

## Étape 2 — Données

Les cellules 3 et 4 écrivent les deux modules du dépôt (`finetune/dictee_to_parquet.py`,
`finetune/prepare_bci_dataset.py` — copies identiques vérifiées par un test). La cellule 5 prépare
`train` (Common Voice + dictée), `dev` et `test`.

In [ ]:
%%writefile dictee_to_parquet.py
"""Convertit l'export dictée (ZIP audiofolder) en dataset parquet Omnilingual — maillon ③ (#474).

Pont entre le maillon ② (export dictée, ADR-0035) et le maillon ③ (fine-tune
Omnilingual, cf. étude de faisabilité docs/benchmarks/0004 et test mécanique 0005).

Entrée : ZIP export dictée
    audio/<x>.webm ...
    metadata.csv : file_name, transcription, language, filiere, text_fr

Sortie : un dataset parquet partitionné (« hive »), seul format que lit le chargeur
`MixtureParquetStorage` de la recette de fine-tune (omnilingual-asr 0.1.0) :
    <racine>/corpus=<corpus>/split=<split>/language=<code>/part-0.parquet
        - text        : transcription normalisée (baoulé) — la CIBLE ASR
        - audio_bytes : audio FLAC 16 kHz mono, en list<int8> (schéma amont)
        - audio_size  : nombre d'échantillons du waveform décodé
    corpus / split / language vivent dans les NOMS DE DOSSIERS, pas dans le fichier :
    la recette découvre les splits et les partitions à partir de l'arborescence.
    Un fichier parquet « à plat » n'est pas lu (aucun split découvert).

Le code langue est celui d'Omnilingual (ISO 639-3 + écriture) : `bci` -> `bci_Latn`.

Usage :
    python dictee_to_parquet.py export.zip dataset_root \\
        --corpus wourri_dictee --split train --language bci \\
        --stats language_distribution_0.tsv

Décodage : soundfile (wav/flac/ogg) ; repli librosa+ffmpeg pour webm/opus/mp3
(le fine-tune tourne sur Colab/Kaggle où ffmpeg est présent). Le rééchantillonnage
utilise librosa si présent, sinon un repli linéaire (dégradé — le vrai run a librosa).

Ce module est recopié tel quel dans le notebook `colab/omnilingual_finetune_smoke_test.ipynb`
(cellule %%writefile) : un test vérifie que les deux copies restent identiques.
"""
from __future__ import annotations

import argparse
import csv
import io
import re
import sys
import unicodedata
import zipfile
from pathlib import Path, PurePosixPath

TARGET_SR = 16000
ROW_GROUP_SIZE = 100  # valeur amont : limite la mémoire du streaming et permet le mélange
FILE_COLUMNS = ["text", "audio_bytes", "audio_size"]

# Apostrophe baoulé : Common Voice bci et la sortie d'Omnilingual l'écrivent U+02BC (ʼ,
# lettre modificative, conservée par la normalisation amont et présente dans le vocabulaire :
# le 1B la produit au benchmark 0003). Les variantes tapées au clavier ENTRE DEUX LETTRES
# (' ’ ‘ ‛) y sont ramenées, sinon un même son aurait plusieurs cibles ; ailleurs ce sont
# des guillemets, retirés comme la ponctuation. Un ton combinant (ɛ̀ = ɛ + U+0300, sans forme
# précomposée) compte comme lettre.
_APOSTROPHES = "'‘’‛"
_INNER_APOSTROPHE = re.compile(rf"(?<=[\ẁ-ͯ])[{_APOSTROPHES}](?=\w)")
_OTHER_APOSTROPHE = re.compile(rf"[{_APOSTROPHES}]")
# Invisibles supprimés par l'amont (shared_deletion_list) + BOM : souvent hérités d'un copier-coller.
_INVISIBLE = re.compile("[​‌‎‏‪‬﻿]")
_PUNCT = re.compile(r"[.,!?;:«»\"“”„()\[\]{}…—–/\\_]")
_DIGIT_WORD = re.compile(r"(?<!\S)\d+(?!\S)")


def normalize_text(text: str) -> str:
    """Normalise une transcription comme la préparation de données amont (text_tools) :
    NFKC, invisibles supprimés, minuscules, ponctuation retirée, mots uniquement numériques
    retirés. Les lettres baoulé (ɛ ɔ ɲ, tons, ʼ) sont conservées : elles portent le sens."""
    t = unicodedata.normalize("NFKC", text or "")
    t = _INVISIBLE.sub("", t).lower()
    t = _INNER_APOSTROPHE.sub("ʼ", t)
    t = _OTHER_APOSTROPHE.sub(" ", t)
    t = _PUNCT.sub(" ", t)
    t = _DIGIT_WORD.sub(" ", t)
    return re.sub(r"\s+", " ", t).strip()


def omni_language_code(code: str) -> str:
    """Code Omnilingual (`lang_ids.py`) = ISO 639-3 + écriture : 'bci' -> 'bci_Latn'.

    Les langues de l'atelier s'écrivent en latin (colonne `script` de la table
    `languages`) ; un code déjà complet ('bci_Latn') est gardé tel quel."""
    code = (code or "").strip()
    if not code:
        raise ValueError("code langue vide")
    return code if "_" in code else f"{code}_Latn"


def decode_to_16k_mono(raw: bytes, filename: str):
    """Octets audio -> (waveform float32 mono 16 kHz, n_samples)."""
    import numpy as np
    import soundfile as sf

    try:
        wav, sr = sf.read(io.BytesIO(raw), dtype="float32", always_2d=True)  # (n, ch)
        wav = wav.mean(axis=1)  # -> mono (n,)
    except Exception:
        wav, sr = _decode_via_librosa(raw, filename)  # webm/opus/mp3 -> ffmpeg
    if sr != TARGET_SR:
        wav = _resample(np.asarray(wav, dtype="float32"), sr, TARGET_SR)
    return np.asarray(wav, dtype="float32"), int(len(wav))


def _decode_via_librosa(raw: bytes, filename: str):
    import tempfile
    import warnings

    import librosa

    suffix = Path(filename).suffix or ".webm"
    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
        tmp.write(raw)
        path = tmp.name
    try:
        # webm : soundfile échoue toujours, librosa passe par audioread + ffmpeg. Ses deux
        # avertissements (« PySoundFile failed », dépréciation d'audioread) sortiraient une fois
        # par clip et noieraient la sortie du notebook (433 clips = 433 paires).
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            warnings.simplefilter("ignore", FutureWarning)
            wav, sr = librosa.load(path, sr=None, mono=True)  # float32 mono
    finally:
        Path(path).unlink(missing_ok=True)
    return wav, sr


def _resample(wav, sr: int, target: int):
    import numpy as np

    try:
        import librosa

        return librosa.resample(wav, orig_sr=sr, target_sr=target)
    except Exception:
        # repli linéaire (le vrai run a librosa ; ce repli garde le script utilisable en test)
        n = int(round(len(wav) * target / sr))
        if n <= 0:
            return wav
        return np.interp(
            np.linspace(0, len(wav) - 1, n), np.arange(len(wav)), wav
        ).astype("float32")


def encode_flac(wav) -> bytes:
    """Waveform float32 mono 16 kHz -> octets FLAC."""
    import soundfile as sf

    buf = io.BytesIO()
    sf.write(buf, wav, TARGET_SR, format="FLAC")
    return buf.getvalue()


def read_export(zip_path) -> list[dict]:
    """Lit le ZIP export dictée -> [{file_name, transcription, language, filiere, text_fr, raw}]."""
    rows: list[dict] = []
    with zipfile.ZipFile(zip_path) as zf:
        names = set(zf.namelist())
        metas = sorted(n for n in names if PurePosixPath(n).name == "metadata.csv")
        if not metas:
            raise SystemExit("metadata.csv introuvable dans le ZIP")
        meta_name = "metadata.csv" if "metadata.csv" in names else metas[0]
        meta = zf.read(meta_name).decode("utf-8-sig")
        for r in csv.DictReader(io.StringIO(meta)):
            fn = (r.get("file_name") or "").strip()
            if not fn:
                continue
            if fn not in names:
                # tolère un préfixe de chemin différent, jamais un autre fichier :
                # même nom exact (pas « 11.webm » pour « 1.webm ») et un seul candidat
                base = PurePosixPath(fn).name
                cands = [n for n in names if PurePosixPath(n).name == base]
                if len(cands) != 1:
                    why = "absent du ZIP" if not cands else f"ambigu ({len(cands)} fichiers)"
                    print(f"  [SKIP] audio {why} : {fn}", file=sys.stderr)
                    continue
                fn = cands[0]
            rows.append(
                {
                    "file_name": fn,
                    "transcription": (r.get("transcription") or "").strip(),
                    "language": (r.get("language") or "").strip(),
                    "filiere": (r.get("filiere") or "").strip(),
                    "text_fr": (r.get("text_fr") or "").strip(),
                    "raw": zf.read(fn),
                }
            )
    return rows


def make_table(examples):
    """[(transcription, waveform float32 mono 16 kHz)] -> table parquet (colonnes FILE_COLUMNS).

    Ignore les paires sans texte après normalisation ou sans audio : une paire
    audio↔texte complète est requise pour l'apprentissage."""
    import numpy as np
    import pyarrow as pa

    texts, audios, sizes = [], [], []
    for text, wav in examples:
        norm = normalize_text(text)
        if not norm or len(wav) == 0:
            continue
        texts.append(norm)
        audios.append(np.frombuffer(encode_flac(wav), dtype=np.int8))
        sizes.append(int(len(wav)))
    return pa.table(
        {
            "text": pa.array(texts, type=pa.string()),
            "audio_bytes": pa.array(audios, type=pa.list_(pa.int8())),
            "audio_size": pa.array(sizes, type=pa.int64()),
        }
    )


def write_partition(table, root, corpus: str, split: str, language: str) -> Path:
    """Écrit la table dans <root>/corpus=…/split=…/language=…/part-0.parquet.

    Remplace les parquet déjà présents dans CETTE partition : un export dictée est
    toujours complet, le reconvertir doit remplacer l'ancien, pas s'y ajouter."""
    import pyarrow.parquet as pq

    for key, value in (("corpus", corpus), ("split", split), ("language", language)):
        if not value or "/" in value or "=" in value:
            raise ValueError(f"{key} invalide pour un nom de partition : {value!r}")
    if "_" in split:
        # la recette lit un split « <split>_<corpus> » comme un filtre de corpus
        raise ValueError(f"split sans '_' attendu (reçu {split!r})")
    if table.num_rows == 0:
        # une partition vide fait échouer la recette au démarrage (lecture de son 1er exemple)
        raise ValueError(f"aucune paire audio↔texte exploitable pour {corpus}/{split}")
    part_dir = Path(root) / f"corpus={corpus}" / f"split={split}" / f"language={language}"
    part_dir.mkdir(parents=True, exist_ok=True)
    for old in part_dir.glob("*.parquet"):
        old.unlink()
    out = part_dir / "part-0.parquet"
    pq.write_table(table, out, row_group_size=ROW_GROUP_SIZE)
    return out


def write_stats(root, out_tsv) -> list[dict]:
    """Écrit le TSV `corpus / language / hours` attendu par `dataset_summary_path`
    (pondération des partitions à l'entraînement). Comme l'outil amont, les heures
    sont cumulées sur tous les splits."""
    import pyarrow.dataset as ds

    table = ds.dataset(str(root), format="parquet", partitioning="hive").to_table(
        columns=["corpus", "language", "audio_size"]
    )
    samples: dict[tuple[str, str], int] = {}
    for corpus, language, size in zip(
        table.column("corpus").to_pylist(),
        table.column("language").to_pylist(),
        table.column("audio_size").to_pylist(),
    ):
        key = (str(corpus), str(language))
        samples[key] = samples.get(key, 0) + int(size)
    stats = [
        {"corpus": c, "language": lang, "hours": n / TARGET_SR / 3600}
        for (c, lang), n in sorted(samples.items())
    ]
    lines = ["corpus\tlanguage\thours"]
    lines += [f"{s['corpus']}\t{s['language']}\t{s['hours']:.6f}" for s in stats]
    Path(out_tsv).parent.mkdir(parents=True, exist_ok=True)
    Path(out_tsv).write_text("\n".join(lines) + "\n", encoding="utf-8")
    return stats


def build_dataset(rows, root, corpus: str, split: str, language: str) -> int:
    """Lignes d'export dictée -> partition parquet. Retourne le nombre de clips écrits."""

    def examples():
        for r in rows:
            if r.get("transcription"):
                wav, _ = decode_to_16k_mono(r["raw"], r["file_name"])
                yield r["transcription"], wav

    table = make_table(examples())
    write_partition(table, root, corpus, split, omni_language_code(language))
    return table.num_rows


def main():
    ap = argparse.ArgumentParser(
        description="Export dictée (ZIP audiofolder) -> dataset parquet Omnilingual (#474)"
    )
    ap.add_argument("zip_path", help="ZIP export dictée (audio/ + metadata.csv)")
    ap.add_argument("out_root", help="racine du dataset (le `data:` de la carte fairseq2)")
    ap.add_argument("--corpus", default="wourri_dictee")
    ap.add_argument("--split", default="train", choices=["train", "dev", "test"])
    ap.add_argument("--language", default=None, help="code langue (ex. bci) ; défaut : celui du CSV")
    ap.add_argument("--stats", default=None, help="écrit aussi le TSV corpus/language/hours")
    args = ap.parse_args()

    rows = read_export(args.zip_path)
    language = args.language
    if not language:
        found = {r["language"] for r in rows if r.get("language")}
        if len(found) != 1:
            raise SystemExit(f"--language requis : langues trouvées dans le CSV = {sorted(found)}")
        language = found.pop()
    n = build_dataset(rows, args.out_root, args.corpus, args.split, language)
    print(f"OK : {n} clips -> {args.out_root} (corpus={args.corpus}, split={args.split}, "
          f"language={omni_language_code(language)})")
    if args.stats:
        write_stats(args.out_root, args.stats)
        print(f"Stats -> {args.stats}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile prepare_bci_dataset.py
"""Prépare le dataset baoulé du fine-tune Omnilingual (#474, ADR-0044).

Common Voice bci (miroir Klayt, révision épinglée du benchmark 0003) :
    train -> split « train », validation -> « dev », test -> « test » (avec --with-test)
+ l'export de l'écran Dictée de l'atelier (optionnel) -> corpus « wourri_dictee », split « train »,
puis le TSV de statistiques et la carte dataset fairseq2.

Le split « test » (290 clips, ceux du benchmark 0003) n'est JAMAIS entraîné : la recette
n'entraîne que sur `train_split: train`. Il est seulement évalué (`valid_split: dev,test`) pour
comparer, dans les mêmes conditions, le modèle de départ et le modèle fine-tuné. La séparation
locuteurs / phrases entre ces splits est vérifiée (docs/benchmarks/0005 §3).

Usage (dans le Python 3.12 du notebook) :
    python prepare_bci_dataset.py --work /workspace --dictee /workspace --with-test

Ce module est recopié tel quel dans les notebooks de finetune/colab/ (cellule %%writefile) :
`python finetune/colab/sync_smoke_test_notebook.py` les resynchronise, un test le vérifie.
"""
from __future__ import annotations

import argparse
import csv
import io
import json
import shutil
import tarfile
import zipfile
from pathlib import Path

import dictee_to_parquet as d2p

DATASET_REPO = "Klayt/baoule-common-voice"
REV = "176d5f8ad6c04e5ca1a0e66770866709bcbb338b"  # même révision que le benchmark 0003
# split Hugging Face -> split de la recette (« dev » : nom attendu par valid_split)
CV_SPLITS = {"train": "train", "validation": "dev", "test": "test"}
CARD_NAME = "wourri_bci"


def get_audio(row):
    """Audio d'une ligne Hugging Face -> (waveform float32 mono, sample rate)."""
    import numpy as np

    a = row["audio"]
    if isinstance(a, dict):  # datasets < 4.0
        return np.asarray(a["array"], dtype="float32"), int(a["sampling_rate"])
    samples = a.get_all_samples()  # datasets >= 4.0 (AudioDecoder)
    return samples.data.numpy().squeeze().astype("float32"), int(samples.sample_rate)


def cv_examples(hf_split, load_dataset=None):
    """[(phrase, waveform 16 kHz)] du split Common Voice demandé."""
    if load_dataset is None:
        from datasets import load_dataset
    for row in load_dataset(DATASET_REPO, revision=REV, split=hf_split):
        wav, sr = get_audio(row)
        if sr != d2p.TARGET_SR:
            wav = d2p._resample(wav, sr, d2p.TARGET_SR)
        yield row["sentence"], wav


def find_export(path, tmp_dir) -> Path:
    """Le ZIP de l'écran Dictée, ou son dossier décompressé (rezippé au format attendu)."""
    src = Path(path)
    if src.is_file():
        return src
    zips = sorted(src.rglob("dictee_*.zip"))
    if zips:
        return zips[0]
    metas = sorted(src.rglob("metadata.csv"))
    if metas:
        base = Path(tmp_dir) / "dictee_export"
        return Path(shutil.make_archive(str(base), "zip", root_dir=metas[0].parent))
    raise SystemExit(f"Ni dictee_*.zip ni metadata.csv sous {path}")


def write_card(asset_dir, data_root) -> Path:
    """Carte fairseq2 `mixture_parquet_asr_dataset` (répertoire FAIRSEQ2_USER_ASSET_DIR)."""
    asset_dir = Path(asset_dir)
    asset_dir.mkdir(parents=True, exist_ok=True)
    card = asset_dir / f"{CARD_NAME}.yaml"
    card.write_text(
        f"name: {CARD_NAME}\n"
        "dataset_family: mixture_parquet_asr_dataset\n"
        "dataset_config:\n"
        f"  data: {data_root}\n"
        "tokenizer_ref: omniASR_tokenizer\n",
        encoding="utf-8",
    )
    return card


def prepare(work, dictee=None, with_test=False, load_dataset=None) -> dict:
    """Construit <work>/wourri_bci (dataset partitionné + TSV) et la carte fairseq2.

    Repart toujours d'un dossier vide : une préparation précédente (autres splits,
    ancien export) ne doit pas se mélanger à celle-ci. Retourne les comptes par partition.
    """
    work = Path(work)
    dataset_dir = work / "wourri_bci"
    data_root = dataset_dir / "version=0"  # le `data:` de la carte fairseq2
    shutil.rmtree(dataset_dir, ignore_errors=True)
    lang = d2p.omni_language_code("bci")
    counts = {}

    for hf_split, split in CV_SPLITS.items():
        if split == "test" and not with_test:
            continue
        table = d2p.make_table(cv_examples(hf_split, load_dataset))
        d2p.write_partition(table, data_root, "cv_bci", split, lang)
        counts[f"cv_bci/{split}"] = table.num_rows

    if dictee:
        export = find_export(dictee, work)
        rows = d2p.read_export(export)
        counts["wourri_dictee/train"] = d2p.build_dataset(rows, data_root, "wourri_dictee", "train", "bci")

    stats = d2p.write_stats(data_root, dataset_dir / "language_distribution_0.tsv")
    card = write_card(work / "fairseq2_assets", data_root)
    return {"counts": counts, "stats": stats, "card": card}


ZENODO_DIR = "bci-datasets"  # racine de l'archive Zenodo 6705861 (data354, CC BY 4.0)


def zenodo_rows(zip_path) -> list[tuple[str, str]]:
    """[(fichier audio, transcription)] de l'archive Zenodo. Les numéros de verset en tête
    (« 28 Amun… ») ne sont pas prononcés (mesuré le 2026-10-24 sur les durées : +0,02 s à
    longueur égale) : `normalize_text` retire les mots numériques, rien d'autre à faire."""
    with zipfile.ZipFile(zip_path) as z:
        tsv = z.read(f"{ZENODO_DIR}/transcription.tsv").decode("utf-8-sig")
    return [(r["id"], r["bci"]) for r in csv.DictReader(io.StringIO(tsv), delimiter="\t") if r.get("bci")]


def zenodo_examples(zip_path, rows):
    with zipfile.ZipFile(zip_path) as z:
        for fichier, texte in rows:
            wav, _ = d2p.decode_to_16k_mono(z.read(f"{ZENODO_DIR}/wav/{fichier}"), fichier)
            yield texte, wav


def cv27_examples(rows, dossier):
    for r in rows:
        raw = (Path(dossier) / "clips" / r["path"]).read_bytes()
        wav, _ = d2p.decode_to_16k_mono(raw, r["path"])
        yield r["sentence"], wav


def extraire_cv27(source, work) -> Path:
    """Dossier `…/bci` de Common Voice 27 : l'archive .tar.gz est extraite une fois dans <work>/cv27."""
    import cv27_split as cvs  # import local : les notebooks du 1er fine-tune n'embarquent pas ce module

    source = Path(source)
    if source.is_file():
        cible = Path(work) / "cv27"
        if not any(cible.rglob("validated.tsv")):
            with tarfile.open(source) as t:
                t.extractall(cible, filter="data")
        source = cible
    return cvs.trouver(source)


def prepare_cv27(work, cv27, zenodo=None, dictee=None) -> dict:
    """2e fine-tune : Common Voice 27 réparti par `cv27_split` (train / dev / test), + Zenodo et
    dictée en train. La répartition est vérifiée contre TOUS les textes d'entraînement
    (aucune phrase du test chez Zenodo ni dans la dictée) avant d'écrire quoi que ce soit."""
    import cv27_split as cvs  # import local : les notebooks du 1er fine-tune n'embarquent pas ce module

    work = Path(work)
    dataset_dir = work / "wourri_bci"
    data_root = dataset_dir / "version=0"
    shutil.rmtree(dataset_dir, ignore_errors=True)
    lang = d2p.omni_language_code("bci")
    zen = zenodo_rows(zenodo) if zenodo else []
    dic = d2p.read_export(find_export(dictee, work)) if dictee else []
    dossier = extraire_cv27(cv27, work)
    rep, manifeste = cvs.preparer(dossier, [t for _, t in zen] + [r["transcription"] for r in dic])
    counts = {}
    for split, rows in rep.items():
        table = d2p.make_table(cv27_examples(rows, dossier))
        d2p.write_partition(table, data_root, "cv_bci", split, lang)
        counts[f"cv_bci/{split}"] = table.num_rows
    if zen:
        table = d2p.make_table(zenodo_examples(zenodo, zen))
        d2p.write_partition(table, data_root, "zenodo_bci", "train", lang)
        counts["zenodo_bci/train"] = table.num_rows
    if dic:
        counts["wourri_dictee/train"] = d2p.build_dataset(dic, data_root, "wourri_dictee", "train", "bci")
    (work / "split_cv27.json").write_text(json.dumps(manifeste, ensure_ascii=False, indent=1), encoding="utf-8")
    stats = d2p.write_stats(data_root, dataset_dir / "language_distribution_0.tsv")
    card = write_card(work / "fairseq2_assets", data_root)
    return {"counts": counts, "stats": stats, "card": card, "splits": manifeste["splits"]}


def main():
    ap = argparse.ArgumentParser(description="Dataset baoulé du fine-tune Omnilingual (#474)")
    ap.add_argument("--work", required=True, help="dossier de travail (sortie)")
    ap.add_argument("--dictee", default=None, help="ZIP de la dictée ou dossier qui le contient")
    ap.add_argument("--with-test", action="store_true",
                    help="ajoute le split test (évalué, jamais entraîné)")
    ap.add_argument("--cv27", default=None, help="Common Voice 27 bci (.tar.gz ou dossier extrait) : 2e fine-tune")
    ap.add_argument("--zenodo", default=None, help="archive Zenodo 6705861 (bci-datasets.zip), avec --cv27")
    args = ap.parse_args()
    if args.cv27:
        result = prepare_cv27(args.work, args.cv27, args.zenodo, args.dictee)
        print("Répartition :", json.dumps(result["splits"], ensure_ascii=False))
    else:
        result = prepare(args.work, args.dictee, args.with_test)
    for name, n in result["counts"].items():
        print(f"{name}: {n} clips")
    for s in result["stats"]:
        print(f"{s['corpus']:15s} {s['language']}  {s['hours']:.2f} h")
    print("Carte :", result["card"])


if __name__ == "__main__":
    main()


In [ ]:
# Cellule 5 — dataset : Common Voice bci (train / dev / test) + dictée d'esse (train)
# DICTEE : dossier où tu as téléversé dictee_bci_433.zip (par défaut /workspace). None = sans dictée.
DICTEE = str(WORK)
args = [VENV_PY, "prepare_bci_dataset.py", "--work", WORK, "--with-test"]
if DICTEE:
    args += ["--dictee", DICTEE]
sh(args)
STATS_TSV = WORK / "wourri_bci" / "language_distribution_0.tsv"
os.environ["FAIRSEQ2_USER_ASSET_DIR"] = str(WORK / "fairseq2_assets")   # carte `wourri_bci`

In [ ]:
# Cellule 6 — configuration (dérivée de la config « recommandation » amont, corrigée)
import copy, yaml

VALID_SPLITS = ["dev", "test"]   # dev : choisir la version ; test : le chiffre à comparer (jamais entraîné)
MAX_NUM_ELEMENTS = 3_840_000     # 240 s d'audio par lot (8×60 s en amont) : ~2 h de données, on limite le surapprentissage
RECO = "omnilingual-asr/workflows/recipes/wav2vec2/asr/configs/ctc-finetune-recommendation.yaml"
with open(RECO, encoding="utf-8") as f:
    BASE = yaml.safe_load(f)

def make_config(num_steps, validate_every, checkpoint_every):
    c = copy.deepcopy(BASE)
    c["model"]["name"] = "omniASR_CTC_1B"
    c["dataset"]["name"] = "wourri_bci"
    c["dataset"]["valid_split"] = ",".join(VALID_SPLITS)
    c["dataset"]["mixture_parquet_storage_config"]["dataset_summary_path"] = str(STATS_TSV)
    c["dataset"]["asr_task_config"]["max_num_elements"] = MAX_NUM_ELEMENTS
    c["dataset"]["asr_task_config"]["example_shuffle_window"] = 1000  # 0 (amont) est refusé par asr_task.py
    # 2 s minimum, comme ctc-finetune.yaml amont (la config « recommandation » n'en met pas). Appliqué au
    # run du 2026-10-19 : les clips plus courts sont exclus à l'identique au départ et à la fin.
    c["dataset"]["asr_task_config"]["min_audio_len"] = 32_000
    c["trainer"]["freeze_encoder_for_n_steps"] = 0                   # entraînement COMPLET
    c["trainer"]["mixed_precision"] = {"mode": "static", "dtype": "torch.bfloat16"}  # A100 : bf16 natif
    c["trainer"]["grad_accumulation"]["num_batches"] = 1
    c["regime"] = {
        "num_steps": num_steps,
        "validate_at_start": True,                 # score du modèle NON entraîné, mêmes conditions
        "validate_after_n_steps": 0,
        "validate_every_n_steps": validate_every,
        "checkpoint_after_n_steps": 0,
        "checkpoint_every_n_steps": checkpoint_every,
        "publish_metrics_after_n_steps": 0,
        "publish_metrics_every_n_steps": 10,
        "save_model_only": True,
    }
    return c

def write_config(tag, cfg):
    path = WORK / f"config_{tag}.yaml"
    path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")
    r = subprocess.run([VENV_PY, "-m", "workflows.recipes.wav2vec2.asr", "--config-file", str(path), "--dump-config"],
                       cwd="omnilingual-asr", capture_output=True, text=True, env=clean_env())
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(f"{tag} : config refusée par fairseq2")
    print(f"{tag} : config acceptée -> {path}")
    return path

write_config("mesure", make_config(100, 50, 100))

In [ ]:
# Cellule 7 — lanceur : journal, pics GPU/RAM, métriques de validation par split, coût
import json, threading

RESULTS_PATH = WORK / "finetune_1b_results.json"

def gpu_memory_mib():
    out = subprocess.run(["nvidia-smi", "-i", "0", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout.strip()
    return int(out) if out.isdigit() else None

_VAL = re.compile(r"Validation Metrics(.*?)\(step (\d+)\) - CTC Loss: ([\d.]+) \| Unit Error Rate \(UER\): "
                  r"([\d.]+) \| Word Error Rate \(WER\): ([\d.]+)")
_TRAIN = re.compile(r"Train Metrics \(step (\d+)\) - CTC Loss: ([\d.]+).*?Total Time: (\d+)s")

def parse_log(text):
    """Validations [{step, split, loss, uer, wer}] et temps d'entraînement {step: secondes cumulées}.

    Le journal fairseq2 contient des codes couleur ANSI invisibles : on les retire avant de lire."""
    text = re.sub(r"\x1b\[[0-9;]*[A-Za-z]", "", text)
    flat = re.sub(r"\s+", " ", text)
    vals, seen = [], {}
    for m in _VAL.finditer(flat):
        label, step = m.group(1).strip(" -()"), int(m.group(2))
        idx = seen.get(step, 0)
        seen[step] = idx + 1
        split = next((s for s in VALID_SPLITS if s in label), None) or (VALID_SPLITS[idx] if idx < len(VALID_SPLITS) else f"#{idx}")
        vals.append({"step": step, "split": split, "loss": float(m.group(3)),
                     "uer": float(m.group(4)), "wer": float(m.group(5))})
    times = {int(s): int(t) for s, _, t in _TRAIN.findall(flat)}
    return vals, times

def seconds_per_step(times):
    steps = sorted(times)
    if len(steps) < 2:
        return None
    return (times[steps[-1]] - times[steps[0]]) / (steps[-1] - steps[0])

def run(tag, config_path):
    out_dir = WORK / f"run_{tag}"
    shutil.rmtree(out_dir, ignore_errors=True)
    log_path = WORK / f"run_{tag}.log"
    env = clean_env({"CUDA_VISIBLE_DEVICES": "0"})
    peak, done = {"gpu": 0}, threading.Event()
    def watch():
        while not done.wait(5):
            peak["gpu"] = max(peak["gpu"], gpu_memory_mib() or 0)
    threading.Thread(target=watch, daemon=True).start()
    t0 = time.time()
    # Affichage filtré (le journal complet reste dans log_path) : scores, erreurs, fin.
    show = re.compile(r"Validation Metrics|Train Metrics|ERROR|Error|Traceback|Task finished|[Cc]heckpoint")
    code = sh([VENV_PY, "-m", "workflows.recipes.wav2vec2.asr", out_dir, "--config-file", config_path],
              cwd="omnilingual-asr", env=env, log_path=log_path, check=False, show=show)
    done.set()
    vals, times = parse_log(log_path.read_text(encoding="utf-8"))
    result = {
        "exit_code": code,
        "minutes": round((time.time() - t0) / 60, 1),
        "peak_gpu_mib": peak["gpu"] or None,
        "seconds_per_step": seconds_per_step(times),
        "checkpoints": sorted({p.name for p in out_dir.rglob("checkpoints/step_*")}, key=lambda s: int(s.split("_")[1])),
        "validations": vals,
        "spent_usd_estimate": round(depense_estimee(), 2),
    }
    results = json.loads(RESULTS_PATH.read_text(encoding="utf-8")) if RESULTS_PATH.exists() else {}
    results[tag] = result
    RESULTS_PATH.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"\n>>> {tag} : code {code}, {result['minutes']} min, {result['seconds_per_step']} s/étape, "
          f"pic GPU {result['peak_gpu_mib']} Mio, checkpoints {result['checkpoints']}, "
          f"dépense estimée {result['spent_usd_estimate']} USD")
    return result

def table(vals):
    print(f"{'étape':>6} | {'split':5} | {'UER (≈CER)':>10} | {'WER':>6}")
    for v in vals:
        print(f"{v['step']:>6} | {v['split']:5} | {v['uer']:>10.2f} | {v['wer']:>6.2f}")

## Étape 3 — MESURE (100 étapes, ~1 h)

Vérifie que l'entraînement complet du 1B tourne sur la machine, et mesure la vitesse. À la fin, la cellule
propose un nombre d'étapes qui tient dans le budget restant.

In [ ]:
# Cellule 8 — MESURE
mesure = run("mesure", WORK / "config_mesure.yaml")
table(mesure["validations"])
sps = mesure["seconds_per_step"]
if mesure["exit_code"] != 0 or not sps:
    raise RuntimeError("La mesure a échoué : envoyer `!tail -60 /workspace/run_mesure.log`")
restant_h = (BUDGET_USD - depense_estimee()) / PRIX_HEURE_USD
utile_h = max(0.0, restant_h * 0.8 - 0.5)        # 20 % de marge + 30 min pour l'évaluation et l'arrêt
N_SUGGERE = int(min(2000, utile_h * 3600 / sps) // 200 * 200)
print(f"\nVitesse : {sps:.1f} s/étape. Budget restant ≈ {restant_h:.1f} h.")
for n in (500, 1000, 1500, 2000):
    print(f"  {n:>5} étapes ≈ {n * sps / 3600:.1f} h ≈ {n * sps / 3600 * PRIX_HEURE_USD:.1f} USD")
print(f">>> Nombre d'étapes proposé dans le budget : {N_SUGGERE}")

## Étape 4 — Entraînement COMPLET

Validation toutes les 100 étapes, sauvegarde toutes les 200. La meilleure version est choisie sur `dev`
(cellule 10), jamais sur `test`.

In [ ]:
# Cellule 9 — COMPLET (laisser N_STEPS = None pour prendre la proposition de la cellule 8)
N_STEPS = None
n = N_STEPS or N_SUGGERE
if n < 200:
    raise RuntimeError(f"{n} étapes : budget insuffisant pour un entraînement utile — ne pas lancer")
complet = run("complet", write_config("complet", make_config(n, 100, 200)))
table(complet["validations"])

In [ ]:
# Cellule 10 — synthèse : meilleure version (choisie sur dev) et gain sur test
def at(vals, step, split):
    return next((v for v in vals if v["step"] == step and v["split"] == split), None)

vals = complet["validations"]
ckpt_steps = [int(c.split("_")[1]) for c in complet["checkpoints"]]
candidats = [s for s in ckpt_steps if at(vals, s, "dev")]
best = min(candidats, key=lambda s: at(vals, s, "dev")["uer"])
t0, tb = at(vals, 0, "test"), at(vals, best, "test")
print(f"Meilleure version (dev) : étape {best} — UER dev {at(vals, best, 'dev')['uer']:.2f}")
print(f"TEST (290 clips, jamais entraînés) — UER : {t0['uer']:.2f} -> {tb['uer']:.2f} | WER : {t0['wer']:.2f} -> {tb['wer']:.2f}")
BEST_DIR = next((WORK / "run_complet").rglob(f"checkpoints/step_{best}"))
results = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))
results["synthese"] = {"best_step": best, "test_uer_depart": t0["uer"], "test_uer_best": tb["uer"],
                       "test_wer_depart": t0["wer"], "test_wer_best": tb["wer"], "best_dir": str(BEST_DIR),
                       "spent_usd_estimate": round(depense_estimee(), 2)}
RESULTS_PATH.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
print("Checkpoint retenu :", BEST_DIR)
print("Résultats :", RESULTS_PATH)

In [ ]:
# Cellule 11 (OPTIONNELLE) — sauvegarder la meilleure version sur Hugging Face (dépôt PRIVÉ)
# Sans cette étape, le modèle disparaît avec la machine (les chiffres, eux, restent dans le JSON).
# Le jeton est demandé ici, jamais écrit sur le disque ni dans les journaux.
HF_REPO = None   # ex. "ouedraogoissouf2012/omniasr-ctc-1b-bci-wourri"
if HF_REPO:
    from getpass import getpass
    token = getpass("Jeton Hugging Face (droit d'écriture) : ")
    code = ("import sys; from huggingface_hub import HfApi; api = HfApi(token=sys.stdin.readline().strip()); "
            f"api.create_repo('{HF_REPO}', private=True, exist_ok=True); "
            f"api.upload_folder(repo_id='{HF_REPO}', folder_path='{BEST_DIR}', path_in_repo='step_{best}'); "
            f"api.upload_file(repo_id='{HF_REPO}', path_or_fileobj='{RESULTS_PATH}', path_in_repo='finetune_1b_results.json'); "
            "print('Envoyé.')")
    sh([VENV_PY, "-c", code], input_text=token + "\n")
    del token

## Fin — ne pas oublier

1. **File → Download** de ce notebook (avec ses sorties) et du fichier `finetune_1b_results.json`.
2. Sur RunPod : **Stop** puis **Terminate** le pod. Tant qu'il existe, il est facturé (et le volume aussi).
3. Reporter les chiffres de la cellule 10 dans `docs/benchmarks/` (fiche de résultats du fine-tune).